## Step 1: Installing Required Libraries for RAG-Based Healthcare Agent

### Objective
To install all necessary dependencies required to build a Retrieval-Augmented Generation (RAG) system using LangChain, ChromaDB, HuggingFace embeddings, and Groq LLM.

### Why This Step is Important
- LangChain enables agentic workflows and tool integration.
- ChromaDB allows persistent vector storage for document retrieval.
- HuggingFace embeddings provide stable, local embedding generation.
- Groq provides high-performance LLM inference.
- Using stable and minimal dependencies avoids runtime conflicts.

This setup ensures a production-ready, stable environment for our healthcare AI agent.


In [ ]:
!pip install -qU \
langchain \
langchain-community \
langchain-chroma \
chromadb \
pypdf \
sentence-transformers \
langchain-groq


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

print("Imports working fine")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Mounted Google Drive.")


## Checking the Folder is created for storing the files and data

In [ ]:
import os

project_path = "/content/drive/MyDrive/health_rag_project"
os.makedirs(project_path, exist_ok=True)

print("Folder ready:", project_path)


## Loading the pdf using the pdf loader in the notebook

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os

pdf1 = os.path.join(project_path, "AHA-GUIDELINEDRIVEN-MANAGEMENT-OF-HYPERTENSION--AN-EVIDENCEBASED-UPDATE.pdf")
pdf2 = os.path.join(project_path, "guidance-on-global-monitoring-for-diabetes.pdf")

loader1 = PyPDFLoader(pdf1)
loader2 = PyPDFLoader(pdf2)

docs1 = loader1.load()
docs2 = loader2.load()

all_docs = docs1 + docs2

print("Total pages loaded:", len(all_docs))


## Creating Chunks of the Data because medical Dataset is heavy

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(all_docs)

print("Total chunks created:", len(chunks))


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings ready")


In [ ]:
from langchain_chroma import Chroma

import shutil
import os

# Store vector DB inside the project path
db_path = os.path.join(project_path, "vector_db")

if os.path.exists(db_path):
    shutil.rmtree(db_path)
    print("Old vector DB deleted.")
else:
    print("No existing DB found.")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_path,
)

print("Vector DB ready")
print("Vector count:", vector_store._collection.count())
print("Expected chunks:", len(chunks))

## Step: Loading API Keys Securely from Google Colab Secrets

### Objective
To securely access API keys stored in Google Colab's Secrets Manager without hardcoding them in the notebook.

### Why This Step is Important
- Prevents exposing API keys in code.
- Keeps notebook secure and shareable.
- Ensures proper environment variable setup for LLM access.

We will retrieve the GROQ_API_KEY from Colab Secrets and set it as an environment variable.


In [ ]:
import os

# Try to read the key from environment first, then fall back to Colab secrets (if available)
groq_key = os.getenv("GROQ_API_KEY")

if not groq_key:
    try:
        from google.colab import userdata  # type: ignore
        groq_key = userdata.get("GROQ_API_KEY")
    except ImportError:
        groq_key = None

if not groq_key:
    raise ValueError(
        "GROQ_API_KEY not found. Set it as an environment variable or Colab secret."
    )

os.environ["GROQ_API_KEY"] = groq_key

print("GROQ key loaded successfully")


In [ ]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq LLM initialized")


## Step: Creating a Retriever from the Vector Database

### Objective
To convert the Chroma vector database into a retriever that can fetch the most relevant document chunks based on a user query.

### Why This Step is Important
- The retriever is the core component of a RAG system.
- It allows the agent to fetch contextually relevant medical information.
- Instead of sending entire documents to the LLM, we send only the most relevant chunks.
- This improves accuracy, reduces hallucination, and increases efficiency.

We will configure the retriever to return the top 3 most relevant chunks for each query.


In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 6
    }
)

## Step: Defining a Retrieval Tool for the Healthcare Agent

### Objective
To create a structured tool that allows the LLM to retrieve relevant medical information from the vector database when required.

### Why This Step is Important
- In an agentic system, the LLM does not directly access the database.
- Instead, it decides when to call a tool.
- The tool fetches relevant medical chunks from the retriever.
- This reduces hallucinations and ensures responses are grounded in official medical documents (WHO & AHA guidelines).

This tool will act as the knowledge access layer of our Healthcare AI Agent.


In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel

class MedicalQuery(BaseModel):
    query: str

def get_medical_guidance(query: str) -> str:
    """
    Retrieves relevant medical information from the vector database.
    """
    results = retriever.invoke(query)

    combined_text = "\n\n".join(
        [doc.page_content[:500] for doc in results]
    )

    return combined_text

medical_tool = StructuredTool.from_function(
    func=get_medical_guidance,
    name="get_medical_guidance",
    description="Search the medical knowledge base for healthcare information.",
    args_schema=MedicalQuery
)

## Step: Binding the Medical Retrieval Tool to the LLM

### Objective
To connect the Groq LLM with the medical retrieval tool so that it can decide when to call the tool during reasoning.

### Why This Step is Important
- This enables tool-calling capability.
- The LLM can dynamically retrieve medical guidance instead of hallucinating answers.
- This transforms a simple chatbot into an agentic system.

We will bind the `get_medical_guidance` tool to the Groq LLM.


In [ ]:
llm_with_tools = llm.bind_tools([get_medical_guidance])

print("LLM successfully bound to medical tool")


## Step: Implementing the Agentic Reasoning Loop

### Objective
To create an interactive reasoning loop where the LLM can:
1. Understand the user's question.
2. Decide whether to call a tool.
3. Execute the tool if needed.
4. Use the retrieved information to generate a grounded response.

### Why This Step is Important
- This creates true agentic behavior.
- The LLM can dynamically fetch knowledge from medical guidelines.
- It reduces hallucination by forcing grounded responses.
- It mimics real-world autonomous reasoning systems.

We will implement a manual tool-calling loop similar to the workshop structure.


In [ ]:
# ==============================
# Healthcare RAG Agent - Final Interactive Version
# ==============================

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# ---- System Prompt ----
system_prompt = SystemMessage(
    content="""
You are a professional healthcare AI assistant.

Rules:
1. Answer ONLY using retrieved medical guidelines.
2. Do NOT add extra general advice.
3. Do NOT generate your own disclaimer.
4. If information is missing, say you do not have enough data.
5. Provide structured medical responses.
"""
)

# ---- Agent Function ----
def run_health_agent(user_query):

    messages = [
        system_prompt,
        HumanMessage(content=user_query),
    ]

    while True:
        response = llm_with_tools.invoke(messages)

        # If LLM calls tool
        if response.tool_calls:
            for tool_call in response.tool_calls:

                tool_name = tool_call["name"]
                tool_args = tool_call["args"]

                if tool_name == "get_medical_guidance":

                    tool_result = get_medical_guidance(tool_args["query"])

                    messages.append(response)
                    messages.append(
                        ToolMessage(
                            content=tool_result,
                            tool_call_id=tool_call["id"],
                        )
                    )
        else:
            final_answer = response.content
            return final_answer


# ---- Interactive Loop ----
print("🏥 Healthcare RAG Agent Ready!")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("👤 You: ")

    if user_input.lower() == "exit":
        print("Agent stopped.")
        break

    response = run_health_agent(user_input)

    print("\n🤖 AI Doctor:\n")
    print(response)
    print("\n" + "="*60 + "\n")